### Supabase DB 및 Gemini 임베딩을 이용한 RAG 데이터 적재 (v1.1.3)
- 로직은 supabase_readme.md 참고
- 필요한 키 : GOOGLE_API_KEY, SUPABASE_URL, SUPABASE_SERVICE_KEY

##### 추후 수정 필요
- source_row_number 컬럼 : csv 통합시 추가 필요(프로토타입은 제외)

In [22]:
# 필요한 패키지 설치 (최초 1회)
# !pip install supabase google-generativeai pandas python-dotenv beautifulsoup4

In [23]:
import os
import google.generativeai as genai
import pandas as pd
import json
import time
from dotenv import load_dotenv
from supabase import create_client, Client

load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_KEY = os.getenv("SUPABASE_SERVICE_KEY")

# 클라이언트 초기화
supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)

print("환경 설정 및 Supabase 클라이언트 초기화 완료")

환경 설정 및 Supabase 클라이언트 초기화 완료


### 1. 데이터 로드 및 전처리

In [24]:
# 1. 데이터 로드
csv_path = "data/clean/combined_normalized_v1_1_3.csv"
df = pd.read_csv(csv_path)

# 2. 임베딩용 텍스트 구성
# v1.1.3 데이터는 이미 summary가 정제되어 있습니다.
def combine_features(row):
    return f"제목: {row['title']}\n카테고리: {row['category']}\n지역: {row['region']}\n대상: {row['target_group']}\n요약: {row['summary']}"

df['combined_text'] = df.apply(combine_features, axis=1)

print(f"총 {len(df)}개의 데이터를 로드했습니다.")
display(df.head(2))

총 30개의 데이터를 로드했습니다.


,source,source_id,title,summary,category,region,provider,target_group,target_age_min,target_age_max,start_date,end_date,detail_url,combined_text
0,biz,PBLN_000000000121111,2026년 물산업 유망 해외프로젝트 발굴 지원사업 모집 공고,민관협력 해외진출 활성화를 위한 '2026년 유망 해외프로젝트 발굴 지원사업'을 다...,수출,전국,기후에너지환경부,중소기업,0,99,2026-04-16,2026-05-08,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...,제목: 2026년 물산업 유망 해외프로젝트 발굴 지원사업 모집 공고\n카테고리: 수...
1,biz,PBLN_000000000121110,2026년 민관협력 해외사업 현지화 지원사업 모집 공고,"우수한 물기술과 역량을 보유한 중소기업과 공공부문이 민관협력체계를 구축하여, 물기업...",수출,전국,기후에너지환경부,중소기업,0,99,2026-04-16,2026-05-08,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...,제목: 2026년 민관협력 해외사업 현지화 지원사업 모집 공고\n카테고리: 수출\n...


### 2. Gemini 임베딩 생성 (gemini-embedding-001 모델, 출력차원을 768차원으로 고정)

In [ ]:
# 1. 임베딩 생성 함수
def get_embedding(text):
    model_name = "models/gemini-embedding-001"
    result = genai.embed_content(
        model=model_name,
        content=text,
        task_type="retrieval_document",
        output_dimensionality=768
    )
    return result['embedding']

def split_text(text, max_length=1500):
    if len(text) <= max_length:
        return [text]
    chunks = []
    for i in range(0, len(text), max_length):
        chunks.append(text[i:i + max_length])
    return chunks

# 2. 경로 및 저장 설정
embedding_dir = "data/embedding"
embedding_file = os.path.join(embedding_dir, "embedded_announcements_v1_2.json")
os.makedirs(embedding_dir, exist_ok=True)

insert_data = []

# 3. 기존 임베딩 파일 확인
if os.path.exists(embedding_file):
    print(f"기존 임베딩 파일을 불러옵니다: {embedding_file}")
    with open(embedding_file, 'r', encoding='utf-8') as f:
        insert_data = json.load(f)
    print(f"성공: {len(insert_data)}개의 데이터를 로드했습니다.")
else:
    print("새로운 임베딩을 생성합니다. (API 호출 발생)")
    
    # [추가] 원본 파일 경로 매핑 사전 정의
    source_file_map = {
        'biz': r'data\raw\biz\bizinfo_page1_size10_20260422_115241.json',
        'kst': r'data\raw\kst\kstartup_page1_size10_20260422_115241.json',
        'youth': r'data\raw\youth\youthcenter_page1_size10_20260422_115241.json'
    }

    for idx, row in df.iterrows():
        full_text = row['combined_text']
        chunks = split_text(full_text, max_length=1500)
        
        for chunk in chunks:
            try:
                embedding = get_embedding(chunk)
                
                # CSV의 모든 컬럼을 가져옴
                data = row.to_dict()
                
                # [수정] 모든 컬럼 전처리 (NaN 및 '확인필요' NULL 처리)
                for key, val in data.items():
                    if val == '확인필요' or pd.isna(val):
                        data[key] = None
                
                # [추가] source_file 매핑 로직 (사전에 정의된 경우만 추가)
                source_val = data.get('source')
                if source_val in source_file_map:
                    data['source_file'] = source_file_map[source_val]
                
                # 필수 데이터 및 임베딩 추가
                data["content"] = chunk
                data["embedding"] = embedding
                
                # 불필요한 임시 컬럼 제외
                if 'combined_text' in data:
                    del data['combined_text']
                
                insert_data.append(data)
                time.sleep(0.5) # rate limit 방지
            except Exception as e:
                print(f"임베딩 생성 오류 (Index {idx}): {e}")
    
    # 파일 저장
    with open(embedding_file, 'w', encoding='utf-8') as f:
        json.dump(insert_data, f, ensure_ascii=False, indent=2)
    print(f"임베딩 데이터 저장이 완료되었습니다: {embedding_file}")

print(f"최종 준비 완료된 청크 개수: {len(insert_data)}")


새로운 임베딩을 생성합니다. (API 호출 발생)
임베딩 데이터 저장이 완료되었습니다: data/embedding\embedded_announcements_v1_1_3.json
최종 준비 완료된 청크 개수: 30


### 3. Supabase `announcements` 테이블에 데이터 적재

In [26]:
def get_supabase_client():
    url = os.getenv("SUPABASE_URL")
    key = os.getenv("SUPABASE_SERVICE_KEY")
    return create_client(url, key)

if insert_data:
    print(f"총 {len(insert_data)}개의 청크 적재를 시작합니다...")
    
    batch_size = 1 # 안정성을 위해 1개씩 처리
    supabase = get_supabase_client()

    for i in range(0, len(insert_data), batch_size):
        batch = insert_data[i:i + batch_size]
        success = False
        
        for retry in range(5):
            try:
                supabase.table("announcements").upsert(batch).execute()
                if (i + 1) % 10 == 0 or (i + 1) == len(insert_data):
                    print(f"[{i+1}/{len(insert_data)}] 적재 진행 중...")
                success = True
                break 
            except Exception as e:
                print(f"⚠️ [{i+1}번 데이터] 오류 발생: {e}")
                supabase = get_supabase_client()
                time.sleep((retry + 1) * 2)
        
        if not success:
            print(f"❌ {i+1}번 데이터 적재 최종 실패.")
            
    print("\n✅ 데이터 적재 프로세스가 완료되었습니다.")

총 30개의 청크 적재를 시작합니다...
[10/30] 적재 진행 중...
[20/30] 적재 진행 중...
[30/30] 적재 진행 중...

✅ 데이터 적재 프로세스가 완료되었습니다.
